In [ ]:

import pandas as pd

df = pd.read_csv('BDD_Billboard_copia_Limpia_2000.csv')
df['date'] = pd.to_datetime(df['date'], errors='coerce')
df['date'] = df['date'].dt.year

df = df.drop(columns=['image_url', 'rank', 'weeks_in_charts', 'last_week'])

# Renombrar columnas
df = df.rename(columns={
    'date': 'Año',
    'song': 'Canción',
    'artist': 'Artista',
    'peak_position': 'Máxima_posición'
})

# Columna 'Canción' ya renombrada
df = df.drop(columns=['Canción'])
df_filtered = df[(df['Máxima_posición'] >= 1) & (df['Máxima_posición'] <= 10)]
df_small = df_filtered[df_filtered['Año'].between(2000, 2010)]


df.head()




,Año,Artista,Máxima_posición
0,2000,Santana Featuring Rob Thomas,1
1,2000,Brian McKnight,2
2,2000,Jessica Simpson,3
3,2000,Whitney Houston,4
4,2000,"Missy ""Misdemeanor"" Elliott Featuring NAS| EVE...",5


In [ ]:
import pandas as pd
import altair as alt
from google.colab import files  # Para subir archivos si no existe el CSV

# --- Cargar archivo ---
try:
    df = pd.read_csv('BDD_Billboard_copia_Limpia_2000.csv')
except FileNotFoundError:
    print("Error: El archivo 'BDD_Billboard_copia_Limpia_2000.csv' no fue encontrado.")
    print("Por favor, súbelo con:")
    print("from google.colab import files; uploaded = files.upload()")
    raise

# --- Preparar datos ---
df['date'] = pd.to_datetime(df['date'], errors='coerce')
df['date'] = df['date'].dt.year

df = df.drop(columns=['image_url', 'rank', 'weeks_in_charts', 'last_week'])

df = df.rename(columns={
    'date': 'Año',
    'song': 'Canción',
    'artist': 'Artista',
    'peak_position': 'Máxima_posición'
})

df = df.drop(columns=['Canción'])
df_filtered = df[(df['Máxima_posición'] >= 1) & (df['Máxima_posición'] <= 10)]
df_small = df_filtered[df_filtered['Año'].between(2000, 2010)]

pivot_table = df_small.pivot_table(
    index='Artista',
    columns='Año',
    values='Máxima_posición',
    aggfunc='count',
    fill_value=0
)

# Top 20 artistas con más canciones en el top 10
top_artistas = pivot_table.sum(axis=1).sort_values(ascending=False).head(20).index
pivot_table_top = pivot_table.loc[top_artistas]

# --- Transformar datos a formato largo ---
df_long = pivot_table_top.reset_index().melt(
    id_vars='Artista',
    var_name='Año',
    value_name='Cantidad'
)

df_long['Año'] = df_long['Año'].astype(str)

# --- Crear selecciones interactivas ---
brush = alt.selection_interval(encodings=['x', 'y'])
highlight = alt.selection_point(fields=['Artista'], empty='none')

# --- Heatmap en tonos morados ---
heatmap = (
    alt.Chart(df_long)
    .mark_rect()
    .encode(
        x=alt.X('Año:O', title='Año'),
        y=alt.Y(
            'Artista:O',
            title='Artista',
            sort=alt.EncodingSortField(field='Cantidad', order='descending')
        ),
        color=alt.condition(
            alt.datum.Cantidad > 0,
            alt.Color('Cantidad:Q',
                      scale=alt.Scale(scheme='purples'),
                      title='Cantidad de canciones'),
            alt.value('white')  # Deja en blanco los espacios sin valor
        ),
        tooltip=['Artista', 'Año', 'Cantidad']
    )
    .properties(
        width=700,
        height=500,
        title='Cantidad de canciones en top 10 por artista y año (2000–2010)'
    )
    .add_params(brush, highlight)
    .transform_filter(brush)
)

# --- Texto dinámico al resaltar artista ---
text = (
    alt.Chart(df_long)
    .mark_text(
        align='left',
        baseline='middle',
        dx=5,
        dy=-5,
        fontWeight='bold',
        color='black'
    )
    .encode(
        x='Año:O',
        y='Artista:O',
        text=alt.condition(highlight, 'Cantidad:Q', alt.value(''))
    )
    .add_params(highlight)
)

# --- Combinar y aplicar estilo con Poppins ---
chart = (heatmap + text).configure(
    title=alt.TitleConfig(font='Poppins', fontSize=18, anchor='start', color='#4527a0'),
    axis=alt.AxisConfig(labelFont='Poppins', titleFont='Poppins', labelFontSize=12, titleFontSize=13),
    legend=alt.LegendConfig(labelFont='Poppins', titleFont='Poppins'),
    view=alt.ViewConfig(stroke='transparent'),  # sin borde
    background='white'  # fondo blanco limpio
)

chart


alt.LayerChart(...)

In [ ]:
import pandas as pd


df = pd.read_csv('BDD_Billboard_copia_Limpia_2000.csv')

# Convertir columna 'date' a datetime y extraer solo el año
df['date'] = pd.to_datetime(df['date'], errors='coerce')
df['date'] = df['date'].dt.year

# Eliminar columnas
df = df.drop(columns=['image_url', 'rank', 'weeks_in_charts', 'last_week'])


df = df.rename(columns={
    'date': 'Año',
    'song': 'Canción',
    'artist': 'Artista',
    'peak_position': 'Máxima_posición'
})

# Eliminar la columna 'Canción'
df = df.drop(columns=['Canción'])

# Filtrar para máximo posición entre 1 y 10
df_filtered = df[(df['Máxima_posición'] >= 1) & (df['Máxima_posición'] <= 10)]

# Dividir en dos dataframes por rangos de años
df_2000_2010 = df_filtered[df_filtered['Año'].between(2000, 2010)]
df_2011_2025 = df_filtered[df_filtered['Año'].between(2011, 2025)]


print("\nDatos 2011-2025:")
print(df_2011_2025.head())


Datos 2011-2025:
        Año                  Artista  Máxima_posición
57400  2011               Bruno Mars                1
57401  2011               Katy Perry                1
57402  2011                    Ke$ha                1
57403  2011  Rihanna Featuring Drake                1
57404  2011                     P!nk                1


In [ ]:
import pandas as pd


df = pd.read_csv('BDD_Billboard_copia_Limpia_2000.csv')

# Convertir columna 'date' a datetime y extraer solo el año
df['date'] = pd.to_datetime(df['date'], errors='coerce')
df['date'] = df['date'].dt.year

# Eliminar columnas
df = df.drop(columns=['image_url', 'rank', 'weeks_in_charts', 'last_week'])


df = df.rename(columns={
    'date': 'Año',
    'song': 'Canción',
    'artist': 'Artista',
    'peak_position': 'Máxima_posición'
})

# Eliminar la columna 'Canción'
df = df.drop(columns=['Canción'])

# Filtrar para máximo posición entre 1 y 10
df_filtered = df[(df['Máxima_posición'] >= 1) & (df['Máxima_posición'] <= 10)]


df_2011_2025 = df_filtered[df_filtered['Año'].between(2011, 2025)]


print("\nDatos 2011-2025:")
print(df_2011_2025.head())


Datos 2011-2025:
        Año                  Artista  Máxima_posición
57400  2011               Bruno Mars                1
57401  2011               Katy Perry                1
57402  2011                    Ke$ha                1
57403  2011  Rihanna Featuring Drake                1
57404  2011                     P!nk                1


In [ ]:
import pandas as pd
import altair as alt

# ✨ Desactivar límites y vegafusion
alt.data_transformers.disable_max_rows()
alt.data_transformers.enable('default')

# === 1. CARGAR Y PROCESAR DATOS ===
df = pd.read_csv('BDD_Billboard_copia_Limpia_2000.csv')

# Convertir columna 'date' a año
df['date'] = pd.to_datetime(df['date'], errors='coerce')
df['Año'] = df['date'].dt.year

# Eliminar columnas innecesarias
df = df.drop(columns=['image_url', 'rank', 'weeks_in_charts', 'last_week'])

# Renombrar columnas
df = df.rename(columns={
    'song': 'Canción',
    'artist': 'Artista',
    'peak_position': 'Máxima_posición'
})

# Eliminar columna 'Canción'
df = df.drop(columns=['Canción'])

# Filtrar canciones con posición máxima entre 1 y 10
df_filtered = df[(df['Máxima_posición'] >= 1) & (df['Máxima_posición'] <= 10)]

# Filtrar rango de años 2011–2025
df_2011_2025 = df_filtered[df_filtered['Año'].between(2011, 2025)]

# === 2. AGRUPAR DATOS ===
pivot_table_top = (
    df_2011_2025
    .groupby(['Artista', 'Año'])
    .size()
    .unstack(fill_value=0)
)

# === 3. TRANSFORMAR DATOS PARA ALTAIR ===
df_long = (
    pivot_table_top
    .reset_index()
    .melt(id_vars='Artista', var_name='Año', value_name='Cantidad')
)

df_long['Año'] = df_long['Año'].astype(str)

# Mostrar solo los 20 artistas más frecuentes
top_artistas = (
    df_long.groupby('Artista')['Cantidad']
    .sum()
    .nlargest(20)
    .index
)
df_long = df_long[df_long['Artista'].isin(top_artistas)]

# === 4. CREAR VISUALIZACIÓN ===

# Selección para zoom
brush = alt.selection_interval(encodings=['x', 'y'])

# Selección para resaltar artista
highlight = alt.selection_point(fields=['Artista'], empty='none')

# --- Heatmap rosa con valores cero en blanco ---
heatmap = (
    alt.Chart(df_long)
    .mark_rect()
    .encode(
        x=alt.X('Año:O', title='Año'),
        y=alt.Y(
            'Artista:O',
            title='Artista',
            sort=alt.EncodingSortField(field='Cantidad', order='descending')
        ),
        color=alt.condition(
            alt.datum.Cantidad > 0,
            alt.Color('Cantidad:Q',
                      scale=alt.Scale(range=['#fde0dd', '#fa9fb5', '#c51b8a']),
                      title='Cantidad de canciones'),
            alt.value('white')  # ← Celdas sin valor quedan blancas
        ),
        tooltip=['Artista', 'Año', 'Cantidad']
    )
    .properties(
        width=700,
        height=500,
        title='Cantidad de entradas en Top 10 por artista y año (2011 - 2025)'
    )
    .add_params(brush, highlight)
    .transform_filter(brush)
)

# --- Texto al hacer clic ---
text = (
    alt.Chart(df_long)
    .mark_text(
        align='left',
        baseline='middle',
        dx=5,
        dy=-5,
        fontWeight='bold',
        color='black'
    )
    .encode(
        x='Año:O',
        y='Artista:O',
        text=alt.condition(highlight, 'Cantidad:Q', alt.value(''))
    )
    .add_params(highlight)
)

# --- Combinar ---
chart = (heatmap + text).configure(
    title=alt.TitleConfig(font='Poppins', fontSize=18, anchor='start', color='#c51b8a'),
    axis=alt.AxisConfig(labelFont='Poppins', titleFont='Poppins', labelFontSize=12, titleFontSize=13),
    legend=alt.LegendConfig(labelFont='Poppins', titleFont='Poppins'),
    view=alt.ViewConfig(stroke='transparent'),  # sin borde
    background='white'  # fondo blanco limpio
)

chart

alt.LayerChart(...)

In [ ]:
!jupyter nbconvert /content/codigo_visualizacion.ipynb --to html --output /content/codigo_visualizacion.html

[NbConvertApp] Converting notebook /content/codigo_visualizacion.ipynb to html
[NbConvertApp] Writing 369676 bytes to /content/codigo_visualizacion.html


In [ ]:
import pandas as pd

# --- Cargar archivo ---
df = pd.read_csv('Album_del_Año.csv')

df.head()


,A�o,Categor�a,Artista,Disco o canci�n,Ganador,G�nero del arista/banda,G�nero disco
0,2024,Album Of The Year,Charli xcx,BRAT,False,M,Dance Pop
1,2024,Album Of The Year,Jacob Collier,Djesse Vol. 4,False,H,Jazz
2,2024,Album Of The Year,Billie Eilish,HIT ME HARD AND SOFT,False,M,Pop
3,2024,Album Of The Year,Chappell Roan,The Rise And Fall Of A Midwest Princess,False,M,Pop
4,2024,Album Of The Year,Taylor Swift,THE TORTURED POETS DEPARTMENT,False,M,Pop


from matplotlib import pyplot as plt
import seaborn as sns
_df_0.groupby('Artista').size().plot(kind='barh', color=sns.palettes.mpl_palette('Dark2'))
plt.gca().spines[['top', 'right',]].set_visible(False)

from matplotlib import pyplot as plt
import seaborn as sns
_df_1.groupby('Disco o canci�n').size().plot(kind='barh', color=sns.palettes.mpl_palette('Dark2'))
plt.gca().spines[['top', 'right',]].set_visible(False)

from matplotlib import pyplot as plt
import seaborn as sns
_df_2.groupby('G�nero del arista/banda').size().plot(kind='barh', color=sns.palettes.mpl_palette('Dark2'))
plt.gca().spines[['top', 'right',]].set_visible(False)

from matplotlib import pyplot as plt
import seaborn as sns
_df_3.groupby('G�nero disco').size().plot(kind='barh', color=sns.palettes.mpl_palette('Dark2'))
plt.gca().spines[['top', 'right',]].set_visible(False)

...

from matplotlib import pyplot as plt
import seaborn as sns
def _plot_series(series, series_name, series_index=0):
  palette = list(sns.palettes.mpl_palette('Dark2'))
  counted = (series['A�o']
                .value_counts()
              .reset_index(name='counts')
              .rename({'index': 'A�o'}, axis=1)
              .sort_values('A�o', ascending=True))
  xs = counted['A�o']
  ys = counted['counts']
  plt.plot(xs, ys, label=series_name, color=palette[series_index % len(palette)])

fig, ax = plt.subplots(figsize=(10, 5.2), layout='constrained')
df_sorted = _df_6.sort_values('A�o', ascending=True)
for i, (series_name, series) in enumerate(df_sorted.groupby('G�nero del arista/banda')):
  _plot_series(series, series_name, i)
  fig.legend(title='G�nero del arista/banda', bbox_to_anchor=(1, 1), loc='upper left')
sns.despine(fig=fig, ax=ax)
plt.xlabel('A�o')
_ = plt.ylabel('count()')

from matplotlib import pyplot as plt
import seaborn as sns
def _plot_series(series, series_name, series_index=0):
  palette = list(sns.palettes.mpl_palette('Dark2'))
  counted = (series['A�o']
                .value_counts()
              .reset_index(name='counts')
              .rename({'index': 'A�o'}, axis=1)
              .sort_values('A�o', ascending=True))
  xs = counted['A�o']
  ys = counted['counts']
  plt.plot(xs, ys, label=series_name, color=palette[series_index % len(palette)])

fig, ax = plt.subplots(figsize=(10, 5.2), layout='constrained')
df_sorted = _df_7.sort_values('A�o', ascending=True)
for i, (series_name, series) in enumerate(df_sorted.groupby('G�nero disco')):
  _plot_series(series, series_name, i)
  fig.legend(title='G�nero disco', bbox_to_anchor=(1, 1), loc='upper left')
sns.despine(fig=fig, ax=ax)
plt.xlabel('A�o')
_ = plt.ylabel('count()')

from matplotlib import pyplot as plt
import seaborn as sns
import pandas as pd
plt.subplots(figsize=(8, 8))
df_2dhist = pd.DataFrame({
    x_label: grp['Disco o canci�n'].value_counts()
    for x_label, grp in _df_8.groupby('Artista')
})
sns.heatmap(df_2dhist, cmap='viridis')
plt.xlabel('Artista')
_ = plt.ylabel('Disco o canci�n')

from matplotlib import pyplot as plt
import seaborn as sns
import pandas as pd
plt.subplots(figsize=(8, 8))
df_2dhist = pd.DataFrame({
    x_label: grp['G�nero del arista/banda'].value_counts()
    for x_label, grp in _df_9.groupby('Disco o canci�n')
})
sns.heatmap(df_2dhist, cmap='viridis')
plt.xlabel('Disco o canci�n')
_ = plt.ylabel('G�nero del arista/banda')

from matplotlib import pyplot as plt
import seaborn as sns
import pandas as pd
plt.subplots(figsize=(8, 8))
df_2dhist = pd.DataFrame({
    x_label: grp['G�nero disco'].value_counts()
    for x_label, grp in _df_10.groupby('G�nero del arista/banda')
})
sns.heatmap(df_2dhist, cmap='viridis')
plt.xlabel('G�nero del arista/banda')
_ = plt.ylabel('G�nero disco')

In [ ]:
import pandas as pd


grammys = pd.read_csv("Album_del_Ano.csv", encoding="latin1")


grammys = grammys.rename(columns={
    'A�o': 'Año',
    'Categor�a': 'Categoría',
    'Disco o canci�n': 'Disco o canción',
    'Ganador': 'Ganador',
    'G�nero del arista/banda': 'Género del artista/banda',
    'G�nero disco': 'Género disco'
})

grammys.head()

,Aï¿½o,Categorï¿½a,Artista,Disco o canciï¿½n,Ganador,Gï¿½nero del arista/banda,Gï¿½nero disco
0,2024,Album Of The Year,Charli xcx,BRAT,False,M,Dance Pop
1,2024,Album Of The Year,Jacob Collier,Djesse Vol. 4,False,H,Jazz
2,2024,Album Of The Year,Billie Eilish,HIT ME HARD AND SOFT,False,M,Pop
3,2024,Album Of The Year,Chappell Roan,The Rise And Fall Of A Midwest Princess,False,M,Pop
4,2024,Album Of The Year,Taylor Swift,THE TORTURED POETS DEPARTMENT,False,M,Pop


In [ ]:
import pandas as pd


billboard = pd.read_csv("BBD_Billboard_copia_Limpieza_2000.csv")
grammys = pd.read_csv("Album_del_Ano.csv")


grammys = grammys.rename(columns={
    'A�o': 'Año',
    'Categor�a': 'Categoría',
    'Disco o canci�n': 'Disco o canción',
    'G�nero del arista/banda': 'Género del artista/banda',
    'G�nero disco': 'Género disco'
})


print(grammys.columns)
grammys.head()

Index(['Año', 'Categoría', 'Artista', 'Disco o canción', 'Ganador',
       'Género del artista/banda', 'Género disco'],
      dtype='object')


,Año,Categoría,Artista,Disco o canción,Ganador,Género del artista/banda,Género disco
0,2024,Album Of The Year,Charli xcx,BRAT,False,M,Dance Pop
1,2024,Album Of The Year,Jacob Collier,Djesse Vol. 4,False,H,Jazz
2,2024,Album Of The Year,Billie Eilish,HIT ME HARD AND SOFT,False,M,Pop
3,2024,Album Of The Year,Chappell Roan,The Rise And Fall Of A Midwest Princess,False,M,Pop
4,2024,Album Of The Year,Taylor Swift,THE TORTURED POETS DEPARTMENT,False,M,Pop


In [ ]:
import pandas as pd


billboard = pd.read_csv("BBD_Billboard_copia_Limpieza_2000.csv")
grammys = pd.read_csv("Album_del_Ano.csv")


grammys = grammys.rename(columns={
    'A�o': 'Año',
    'Categoría': 'Categoría',
    'Artista': 'Artista',
    'Disco o canción': 'Disco o canción',
    'Ganador': 'Ganador',
    'Género del artista/banda': 'Género del artista/banda',
    'Género disco': 'Género disco'
})


billboard = billboard.rename(columns={
    'artist': 'Artista',
    'peak_position': 'Peak',
    'song': 'Canción Billboard',
    'year': 'Año'
})


if "Año" not in billboard.columns:
    billboard["Año"] = pd.to_datetime(billboard["date"]).dt.year


def limpiar_texto(x):
    if isinstance(x, str):
        x = x.lower().strip()
        x = x.replace("&", "and")
        x = x.replace("feat.", "featuring").replace("feat", "featuring")
        x = x.replace("ft.", "featuring").replace("ft", "featuring")
        return x
    return x

billboard["Artista_norm"] = billboard["Artista"].apply(limpiar_texto)
grammys["Artista_norm"]   = grammys["Artista"].apply(limpiar_texto)


billboard_top10 = billboard[billboard["Peak"] <= 10]


union = billboard_top10.merge(
    grammys,
    on=["Año", "Artista_norm"],
    how="inner",
    suffixes=(" Billboard", " Grammy")
)


resultado = union[[
    "Año",
    "Artista Billboard",
    "Canción Billboard",
    "Peak"
]]

print(resultado)
resultado.head(20)

       Año  Artista Billboard     Canción Billboard  Peak
0     2000             Eminem   The Real Slim Shady     1
1     2000             Eminem   The Real Slim Shady     7
2     2000             Eminem   The Real Slim Shady     6
3     2000             Eminem   The Real Slim Shady     4
4     2000             Eminem   The Real Slim Shady     4
...    ...                ...                   ...   ...
3638  2024  Sabrina Carpenter  Please Please Please     1
3639  2024      Billie Eilish    Birds Of A Feather     2
3640  2024  Sabrina Carpenter                 Taste     2
3641  2024  Sabrina Carpenter              Espresso     3
3642  2024      Chappell Roan      Good Luck; Babe!     4

[3643 rows x 4 columns]


,Año,Artista Billboard,Canción Billboard,Peak
0,2000,Eminem,The Real Slim Shady,1
1,2000,Eminem,The Real Slim Shady,7
2,2000,Eminem,The Real Slim Shady,6
3,2000,Eminem,The Real Slim Shady,4
4,2000,Eminem,The Real Slim Shady,4
5,2000,Eminem,The Real Slim Shady,4
6,2000,Eminem,The Real Slim Shady,4
7,2000,Eminem,The Real Slim Shady,4
8,2000,Eminem,The Real Slim Shady,4
9,2000,Eminem,The Real Slim Shady,4


In [6]:
!pip install Unidecode

import pandas as pd
import altair as alt
import unidecode
import numpy as np


album = pd.read_csv("Album_del_Ano.csv")
billboard = pd.read_csv("BBD_Billboard_copia_Limpieza_2000.csv")


album.columns = [
    "Año", "Categoria", "Artista", "Disco_o_cancion",
    "Ganador", "Genero_artista", "Genero_disco"
]


billboard["Año"] = pd.to_datetime(billboard["date"], errors="coerce").dt.year


album["Artista_norm"] = album["Artista"].apply(lambda x: unidecode.unidecode(str(x)).lower())
billboard["artist_norm"] = billboard["artist"].apply(lambda x: unidecode.unidecode(str(x)).lower())


artistas_fijos = [
    "Adele", "Billie Eilish", "Bruno Mars", "Harry Styles",
    "Mumford & Sons", "Norah Jones", "Outkast", "Taylor Swift", "U2"
]
artistas_fijos_norm = [unidecode.unidecode(a).lower() for a in artistas_fijos]


subset = billboard[billboard["artist_norm"].isin(artistas_fijos_norm)].copy()


ganadores = album[album["Ganador"] == 1][["Artista", "Año", "Disco_o_cancion"]].copy()
ganadores["Artista_norm"] = ganadores["Artista"].apply(lambda x: unidecode.unidecode(str(x)).lower())


años_ganados = (
    ganadores.groupby("Artista_norm")["Año"]
    .apply(lambda x: ", ".join(x.astype(str)))
    .to_dict()
)


subset["años_ganó_album_del_año"] = subset["artist_norm"].map(años_ganados)


np.random.seed(1)
subset["y_jitter"] = (
    subset.groupby("artist").cumcount() * 0.25
    + np.random.uniform(-0.1, 0.1, len(subset))
)


chart = (
    alt.Chart(subset)
    .mark_circle(filled=True)
    .encode(
        x=alt.X("Año:O", title="Año"),
        y=alt.Y("artist:N", title="Artista"),
        yOffset="y_jitter:Q",

        size=alt.Size(
            "peak_position:Q",
            scale=alt.Scale(domain=[1, 100], range=[2000, 20]),
            title="Peak Billboard (1 = mejor)"
        ),

        color=alt.Color(
            "artist:N",
            title="Artista",
            scale=alt.Scale(
                range=[
                    "#EAC119", "#808BC5", "#EAA7C7", "#9ED6DF", "#245E55",
                    "#ED773C", "#C63F3E", "#1D1D1B", "#D8639C", "#EDD470",
                    "#49B5A3", "#444E7D", "#24B2C9", "#DA7676", "#F1A781"
                ]
            )
        ),

        tooltip=[
            "artist:N",
            "Año:O",
            "peak_position:Q",
            alt.Tooltip("años_ganó_album_del_año:N", title="Ganó Álbum del Año en"),
        ]
    )
    .properties(
        width=850,
        height=550,
        title="Peaks en la lista Billboard de los ganadores a Álbum del Año"
    )
)

chart


alt.Chart(...)

In [11]:
from google.colab import files
files.download('GráficosTop10.html')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>